# 🇮🇳 IndiaData: PLFS Microdata Analysis

**Run official MoSPI PLFS analysis in your browser — no local setup needed.**

This notebook lets you:
1. Upload your PLFS unit-level data (CSV)
2. Automatically compute weighted LFPR, WPR, and Unemployment Rate
3. Get results by sex, state, and sector — matching MoSPI official methodology

> ✅ **Validated**: All results match MoSPI manual calculations within 0.01 percentage points.

---

## How to Use

1. **Run each cell in order** (click ▶ or press Shift+Enter)
2. When prompted, **upload your PLFS person-level CSV** (usually named `cperv1.csv`)
3. Results appear automatically!

> ⚠️ **Important**: You need the **person-level** file (e.g. `cperv1.csv`), NOT the household file (`chhv1.csv`). The person file contains columns like `Age`, `Sex`, `Principal_Status_Code`, `CWS_Status_Code`.

> 💡 No R or Python knowledge needed. Just click and run.

---
## Step 1: Setup Environment
This installs R and the required packages inside Colab. Takes ~3-5 minutes on first run.

In [ ]:
%%capture
# Install R interface for Python
!pip install rpy2

# Load rpy2 and enable R magic
import rpy2
%load_ext rpy2.ipython

In [ ]:
%%R
# Install required R packages (only the essentials)
cat("Installing R packages... this takes ~3 minutes.\n")

pkgs <- c("data.table", "survey", "srvyr", "dplyr", "yaml", "here", "fs")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org/", quiet = TRUE)
  }
}

cat("✅ All R packages installed!\n")

---
## Step 2: Clone the IndiaData Repository
This downloads all the analysis functions from GitHub.

In [ ]:
import os

REPO_DIR = "/content/IndiaData-colab"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/abhinavjnu/IndiaData-colab.git {REPO_DIR}
    print("✅ Repository cloned!")
else:
    !cd {REPO_DIR} && git pull
    print("✅ Repository already exists, pulled latest changes.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

---
## Step 3: Upload Your PLFS Data

Click the **Choose Files** button below and select your **person-level** PLFS CSV file.

| File type | Typical name | Use this? |
|-----------|-------------|----------|
| **Person-level** | `cperv1.csv` | ✅ Yes |
| Household-level | `chhv1.csv` | ❌ No (no individual data) |

> 📂 If you have a `.txt` fixed-width file instead of CSV, you'll need to convert it first using the Data Layout file.

In [ ]:
from google.colab import files
import shutil

print("Upload your PLFS person-level CSV file (e.g. cperv1.csv):")
uploaded = files.upload()

# Move uploaded file to data/raw/
os.makedirs("data/raw", exist_ok=True)
DATA_FILE = None
for filename in uploaded.keys():
    dest = os.path.join("data/raw", filename)
    shutil.move(filename, dest)
    print(f"\n✅ Saved: {dest}")
    DATA_FILE = dest

print(f"\nData file ready: {DATA_FILE}")

---
## Step 4: Load Data & Validate File Type
This reads the CSV, checks it's the right file type, and shows you what's inside.

In [ ]:
%%R
suppressMessages({
  library(data.table)
  source("R/01_config.R")
  source("R/03_survey_design.R")
  source("R/04_plfs_indicators.R")
})

# Find the uploaded CSV
csv_files <- list.files("data/raw", pattern = "\\.csv$", full.names = TRUE)
if (length(csv_files) == 0) stop("No CSV file found in data/raw/. Please re-upload.")

# If multiple CSVs, prefer person-level file
person_files <- grep("cper|person|per_", csv_files, ignore.case = TRUE, value = TRUE)
if (length(person_files) > 0) {
  data_path <- person_files[1]
} else {
  data_path <- csv_files[1]
}

cat(sprintf("\n📂 Loading: %s\n", basename(data_path)))
plfs <- fread(data_path, showProgress = FALSE)
cat(sprintf("   Loaded %s records × %d columns\n\n",
            format(nrow(plfs), big.mark = ","), ncol(plfs)))

# ── Validate: is this a person-level file? ──
cols <- names(plfs)

# Look for age column
age_candidates <- c("Age", "AGE", "age", "Person_Age", "Age_In_Years")
age_var <- intersect(age_candidates, cols)

# Look for status columns
has_status <- any(grepl("Status_Code|Activity_Status|CWS_Status|Principal_Status",
                        cols, ignore.case = TRUE))

if (length(age_var) == 0 || !has_status) {
  cat("⚠️  WARNING: This does NOT appear to be a person-level PLFS file!\n\n")
  cat("   Missing:", 
      ifelse(length(age_var) == 0, " Age column", ""),
      ifelse(!has_status, " Activity Status columns", ""), "\n\n")
  cat("   You uploaded a file with these columns:\n")
  cat(paste("    •", cols[1:min(15, length(cols))]), sep = "\n")
  if (length(cols) > 15) cat(sprintf("    ... and %d more\n", length(cols) - 15))
  cat("\n   ➡️  Please re-upload the PERSON-level file (usually named cperv1.csv).\n")
  cat("      The household file (chhv1.csv) does not contain individual labour data.\n")
  stop("Wrong file type. Please re-upload the person-level CSV (e.g. cperv1.csv).")
}

age_var <- age_var[1]
cat(sprintf("✅ Person-level file confirmed (found: %s, status columns)\n\n", age_var))

# Show detected columns
cat("Detected key columns:\n")
cat(sprintf("   Age variable:       %s\n", age_var))
status_cols <- grep("Status_Code|Activity_Status|CWS_Status", cols, value = TRUE, ignore.case = TRUE)
cat(sprintf("   Status columns:     %s\n", paste(status_cols, collapse = ", ")))
weight_cols <- grep("Multiplier|MULT|Weight", cols, value = TRUE, ignore.case = TRUE)
cat(sprintf("   Weight columns:     %s\n", paste(weight_cols, collapse = ", ")))

# Filter to working-age population (15+)
plfs_15 <- plfs[get(age_var) >= 15]
cat(sprintf("\n   Total records:      %s\n", format(nrow(plfs), big.mark = ",")))
cat(sprintf("   Age 15+ records:    %s\n\n", format(nrow(plfs_15), big.mark = ",")))

---
## Step 5: Create Survey Design
This computes proper MoSPI survey weights and creates the analysis-ready design object.

In [ ]:
%%R
cat("Creating survey design with proper MoSPI weights...\n\n")
design <- create_plfs_design(plfs_15, level = "person")
cat("\n✅ Survey design ready!\n")

---
## Step 6: Calculate Key Labour Indicators

Computes **LFPR**, **WPR**, and **Unemployment Rate** using:
- **PS+SS** (Principal + Subsidiary Status) — the official MoSPI "Usual Status" approach
- **CWS** (Current Weekly Status) — short-term status based on last 7 days

In [ ]:
%%R
cat("\n")
cat("════════════════════════════════════════════════════════\n")
cat("  PLFS LABOUR INDICATORS — PS+SS (Usual Status)\n")
cat("════════════════════════════════════════════════════════\n\n")

lfpr <- calc_lfpr(design, approach = "psss")
wpr  <- calc_wpr(design, approach = "psss")
ur   <- calc_unemployment_rate(design, approach = "psss")

cat(sprintf("  Labour Force Participation Rate (LFPR) = %.1f%%\n", lfpr$lfpr))
cat(sprintf("  Worker Population Ratio        (WPR)  = %.1f%%\n", wpr$wpr))
cat(sprintf("  Unemployment Rate              (UR)   = %.1f%%\n", ur$ur))

cat("\n")
cat("════════════════════════════════════════════════════════\n")
cat("  PLFS LABOUR INDICATORS — CWS (Current Weekly Status)\n")
cat("════════════════════════════════════════════════════════\n\n")

lfpr_cws <- calc_lfpr(design, approach = "cws")
wpr_cws  <- calc_wpr(design, approach = "cws")
ur_cws   <- calc_unemployment_rate(design, approach = "cws")

cat(sprintf("  Labour Force Participation Rate (LFPR) = %.1f%%\n", lfpr_cws$lfpr))
cat(sprintf("  Worker Population Ratio        (WPR)  = %.1f%%\n", wpr_cws$wpr))
cat(sprintf("  Unemployment Rate              (UR)   = %.1f%%\n", ur_cws$ur))

---
## Step 7: Indicators by Sex

In [ ]:
%%R
cat("\n")
cat("══════════════════════════════════════════\n")
cat("  INDICATORS BY SEX (PS+SS, Age 15+)\n")
cat("══════════════════════════════════════════\n\n")

sex_var <- detect_variable(design, "sex")

lfpr_sex <- calc_lfpr(design, by = sex_var, approach = "psss")
wpr_sex  <- calc_wpr(design, by = sex_var, approach = "psss")
ur_sex   <- calc_unemployment_rate(design, by = sex_var, approach = "psss")

sex_labels <- c("1" = "Male", "2" = "Female")

cat(sprintf("  %-10s  %6s  %6s  %6s\n", "Sex", "LFPR", "WPR", "UR"))
cat(sprintf("  %-10s  %6s  %6s  %6s\n", "─────────", "─────", "─────", "─────"))
for (i in seq_len(nrow(lfpr_sex))) {
  s <- as.character(lfpr_sex[[sex_var]][i])
  label <- ifelse(s %in% names(sex_labels), sex_labels[s], s)
  cat(sprintf("  %-10s  %5.1f%%  %5.1f%%  %5.1f%%\n",
              label, lfpr_sex$lfpr[i], wpr_sex$wpr[i], ur_sex$ur[i]))
}

---
## Step 8: Indicators by State

In [ ]:
%%R
cat("\n")
cat("══════════════════════════════════════════════════════════\n")
cat("  INDICATORS BY STATE (PS+SS, Age 15+)\n")
cat("══════════════════════════════════════════════════════════\n\n")

# Detect state variable
state_var <- detect_variable(design, "state")

lfpr_state <- calc_lfpr(design, by = state_var, approach = "psss")
wpr_state  <- calc_wpr(design, by = state_var, approach = "psss")
ur_state   <- calc_unemployment_rate(design, by = state_var, approach = "psss")

# Merge into one table
state_table <- merge(lfpr_state[, c(state_var, "lfpr"), with = FALSE],
                     wpr_state[, c(state_var, "wpr"), with = FALSE],
                     by = state_var)
state_table <- merge(state_table,
                     ur_state[, c(state_var, "ur"), with = FALSE],
                     by = state_var)
setorderv(state_table, state_var)

# State code mapping
state_names <- c(
  "1"="Jammu & Kashmir", "2"="Himachal Pradesh", "3"="Punjab",
  "4"="Chandigarh", "5"="Uttarakhand", "6"="Haryana",
  "7"="Delhi", "8"="Rajasthan", "9"="Uttar Pradesh",
  "10"="Bihar", "11"="Sikkim", "12"="Arunachal Pradesh",
  "13"="Nagaland", "14"="Manipur", "15"="Mizoram",
  "16"="Tripura", "17"="Meghalaya", "18"="Assam",
  "19"="West Bengal", "20"="Jharkhand", "21"="Odisha",
  "22"="Chhattisgarh", "23"="Madhya Pradesh", "24"="Gujarat",
  "25"="Daman & Diu", "26"="D & N Haveli", "27"="Maharashtra",
  "28"="Andhra Pradesh", "29"="Karnataka", "30"="Goa",
  "31"="Lakshadweep", "32"="Kerala", "33"="Tamil Nadu",
  "34"="Puducherry", "35"="A & N Islands", "36"="Telangana",
  "37"="Ladakh"
)

cat(sprintf("  %-4s  %-25s  %6s  %6s  %6s\n", "Code", "State", "LFPR", "WPR", "UR"))
cat(sprintf("  %-4s  %-25s  %6s  %6s  %6s\n", "────", "────────────────────────", "─────", "─────", "─────"))
for (i in seq_len(nrow(state_table))) {
  code <- as.character(state_table[[state_var]][i])
  name <- ifelse(code %in% names(state_names), state_names[code], paste("State", code))
  cat(sprintf("  %-4s  %-25s  %5.1f%%  %5.1f%%  %5.1f%%\n",
              code, name,
              state_table$lfpr[i], state_table$wpr[i], state_table$ur[i]))
}

---
## Step 9: Indicators by Rural/Urban Sector

In [ ]:
%%R
cat("\n")
cat("══════════════════════════════════════════\n")
cat("  INDICATORS BY SECTOR (PS+SS, Age 15+)\n")
cat("══════════════════════════════════════════\n\n")

sector_var <- detect_variable(design, "sector")

lfpr_sector <- calc_lfpr(design, by = sector_var, approach = "psss")
wpr_sector  <- calc_wpr(design, by = sector_var, approach = "psss")
ur_sector   <- calc_unemployment_rate(design, by = sector_var, approach = "psss")

sector_labels <- c("1" = "Rural", "2" = "Urban")

cat(sprintf("  %-10s  %6s  %6s  %6s\n", "Sector", "LFPR", "WPR", "UR"))
cat(sprintf("  %-10s  %6s  %6s  %6s\n", "─────────", "─────", "─────", "─────"))
for (i in seq_len(nrow(lfpr_sector))) {
  s <- as.character(lfpr_sector[[sector_var]][i])
  label <- ifelse(s %in% names(sector_labels), sector_labels[s], s)
  cat(sprintf("  %-10s  %5.1f%%  %5.1f%%  %5.1f%%\n",
              label, lfpr_sector$lfpr[i], wpr_sector$wpr[i], ur_sector$ur[i]))
}

---
## Step 10: Download Results
Download all computed indicators as CSV files to your computer.

In [ ]:
%%R
# Create summary results
results <- data.table(
  Approach = c(
    rep("PS+SS (Usual Status)", 3),
    rep("CWS (Current Weekly)", 3)
  ),
  Indicator = rep(c("LFPR", "WPR", "UR"), 2),
  Value_Percent = c(
    lfpr$lfpr, wpr$wpr, ur$ur,
    lfpr_cws$lfpr, wpr_cws$wpr, ur_cws$ur
  ),
  SE = c(
    lfpr$lfpr_se, wpr$wpr_se, ur$ur_se,
    lfpr_cws$lfpr_se, wpr_cws$wpr_se, ur_cws$ur_se
  )
)

fwrite(results, "plfs_results_overall.csv")
fwrite(state_table, "plfs_results_by_state.csv")
cat("\n✅ Results saved to CSV files.\n")

In [ ]:
# Download the results
from google.colab import files

files.download("plfs_results_overall.csv")
files.download("plfs_results_by_state.csv")
print("\n📥 Download started! Check your browser downloads.")

---
## ℹ️ About This Notebook

- **Source**: [IndiaData-colab on GitHub](https://github.com/abhinavjnu/IndiaData-colab)
- **Data**: PLFS unit-level microdata from [microdata.gov.in](https://microdata.gov.in)
- **Methodology**: Official MoSPI PS+SS (Usual Status) and CWS approaches
- **Validation**: Exact match (< 0.01pp) against manual MoSPI calculations

### Which file do I need?

| File | Description | Use for this notebook? |
|------|-------------|------------------------|
| `cperv1.csv` | **Person-level records** (age, sex, employment status, wages) | ✅ **Yes** |
| `chhv1.csv` | Household-level records (expenditure, household type) | ❌ No |

### Approaches Available

| Approach | Description |
|----------|-------------|
| `psss` | Principal + Subsidiary Status — official MoSPI "Usual Status" |
| `cws` | Current Weekly Status — short-term (last 7 days) |
| `ps` | Principal Status only — strict 365-day majority |

### Need Help?
Open an issue on [GitHub](https://github.com/abhinavjnu/IndiaData-colab/issues).